# Agentic RL

# ReAct

熟悉下Agent运行过程，方便后面学习

In [1]:
import json
import os
import subprocess
from openai import OpenAI

# 初始化 OpenAI 客户端（从环境变量读取配置）
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_BASE_URL"),
)

# ==============================================================================
# ① 工具定义 (Tools Description / Schema)
# - 目的：这不是真正的代码执行，而是用 JSON Schema 格式写给大模型的“工具菜单与说明书”。
# - 原理：API 会把这段声明转成 System Prompt 丢给模型，模型读取参数类型后，
#         才知道当它需要某种能力时，应该按什么 JSON 格式吐出参数。
# ==============================================================================
tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_bash",
            "description": "Execute a bash command and return output",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read content of a file",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    },
]


# ==============================================================================
# ② 环境与执行层 (Environment / Action Execution)
# - 目的：扮演 Agent 的“手”。大模型本身只是个概率预测神经网络，没有任何系统操作权限。
# - 原理：当模型提出“想调用某工具”的意图后，Python 代码接管控制权，在本地真正跑命令/读文件，
#         并提取控制台的输出结果（Observation/观测值），这是 POMDP 中的状态转移过程。
# ==============================================================================
def execute_tool(name, args):
    if name == "execute_bash":
        # 真正调用操作系统的 shell 执行命令，捕获标准输出 (stdout) 和标准错误 (stderr)
        r = subprocess.run(
            args["command"], shell=True, capture_output=True, text=True
        )
        return r.stdout + r.stderr
    elif name == "read_file":
        # 真正的读取本地文件内容
        with open(args["path"]) as f:
            return f.read()
    return f"Unknown tool: {name}"


# ==============================================================================
# ③ Agent 核心主循环 (ReAct Loop: Sense -> Think -> Act -> Observe)
# - 目的：维护完整的上下文对话历史 `messages`，驱动模型多轮自适应决策，直到得出最终答案。
# ==============================================================================
def run_agent(task, max_turns=5):
    # 初始化历史消息队列：把系统设定与用户的真实 Task 作为起点塞入上下文
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": task},
    ]

    # 限制最大轮次 (max_turns)，防止模型陷入死循环跑满资源
    for turn in range(max_turns):
        # ----------------------------------------------------------------------
        # 步骤 A [感知与思考 (Sense & Think)]：
        # 将当前的全部历史上下文 + 工具菜单送给 LLM。LLM 会根据当前上下文判断：
        # 1. 直接输出文本回答（无需调工具）；2. 生成 tool_calls 请求（需调工具）。
        # ----------------------------------------------------------------------
        response = client.chat.completions.create(
            model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message

        # 【极其关键】：必须把 LLM 刚才吐出的回答（包括它的思考或 tool_calls 请求）
        # 追加进 messages 历史。如果不追加，下一轮 LLM 将丢失自己“上一轮调了什么工具”的记忆！
        messages.append(msg)

        # ----------------------------------------------------------------------
        # 步骤 B [终止判定]：
        # 如果模型返回的消息中没有 tool_calls，说明它认为信息已足够，得出了最终结论。
        # 循环结束，直接将文本答案返回给用户。
        # ----------------------------------------------------------------------
        if not msg.tool_calls:
            return msg.content

        # ----------------------------------------------------------------------
        # 步骤 C [行动与观测 (Act & Observe)]：
        # 如果 msg.tool_calls 不为空，说明模型提出了工具调用请求。
        # 遍历模型想要调用的每一个工具，用本地 Python 脚本代为执行。
        # ----------------------------------------------------------------------
        for tc in msg.tool_calls:
            # 1. 解析模型给出的参数 JSON 字符串（如 {"command": "ls *.md"}）
            args = json.loads(tc.function.arguments)
            print(f"  [Turn {turn+1}] 调用工具: {tc.function.name}({args})")

            # 2. 真正交由本地环境执行，拿到运行结果 (Observation)
            result = execute_tool(tc.function.name, args)

            # 3. 【极其关键】：把工具运行的结果伪装成 role="tool" 的消息塞回 messages 历史。
            #    注意：必须带上 `tool_call_id`，LLM 才知道“这个执行结果”对应的是它“刚才发出的哪一个请求”。
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

        # 步骤 D：带着包含了【新增 tool 运行结果】的全新 messages 历史，自动进入下一轮 for 循环！

    return "（达到最大轮次，强制停止）"


# ==============================================================================
# 入口：发起任务
# 数据流过程：
# 1. [Turn 1]: LLM 收到 Prompt -> 判断需要列出文件 -> 生成 tool_calls ("ls *.md")
# 2. Python 捕获该请求 -> 跑 shell 得到 "README.md\n" -> 以 role="tool" 追加进上下文
# 3. [Turn 2]: LLM 拿到上下文里的文件列表 -> 算出一共有 1 个文件 -> 吐出最终文本答案 -> 退出循环
# ==============================================================================
print(run_agent("查看当前目录下有哪些 .md 文件，告诉我一共有几个"))

ModuleNotFoundError: No module named 'openai'

# Agentic RL 核心

代码里用四类标签把这个闭环写出来：

- `<think>...</think>`：模型内部推理，模型生成，参与训练
- `<search>query</search>`：模型 action，模型生成，参与训练
- `<information>docs</information>`：环境 observation，检索器返回，masked 不训练
- `<answer>final answer</answer>`：模型最终回答，模型生成，参与训练

一次 rollout 的核心逻辑很短：模型生成到 `</search>` 时暂停，系统解析 query，调用检索器，把返回文档包进 `<information>` 后拼回上下文；模型读到这些 observation 后继续生成，直到输出 `<answer>` 或达到最大轮数。

最重要的训练细节是 mask。`<search>`、`<think>`、`<answer>` 是模型生成的 token，可以被优化；`<information>` 是检索器返回的文本，只能作为上下文，不应该让模型学习“生成搜索结果”。Search-R1 里的 `state_masking=true` 对应的就是这件事。

SFT 和 RL 在 Agentic 场景中的分工：

1. SFT 教格式：教会模型工具调用的语法、基本的交互协议。
2. RL 教策略：教会模型何时调用工具、如何组合多步行动、失败后如何恢复。